# Module 1.6 — RAGAS in Practice

Module 0 introduced RAGAS as one of the four tools in this tutorial's landscape — a RAG-specific metric library, narrower in scope than DeepEval but a common default in the RAG ecosystem, worth knowing on its own terms. Modules 1.2–1.3 already showed one DeepEval-RAGAS crossover (`RAGASAnswerRelevancyMetric`, a DeepEval binding into RAGAS). This notebook uses RAGAS **directly**, in its own API shape, and closes with the batch/CI pattern used to run it against a live system at scale.

_Source: `labs/rag-evaluation/ragas-scripts/SingleTurnSample.ipynb`, `labs/rag-evaluation/ragas-scripts/Faithfulness.py`, and `labs/rag-evaluation/ragas-scripts/Evaluate_RAG_PyTest.py` — the folder also contains a small FastAPI/Streamlit RAG bot (`fastapi_rag_bot/`) used as the eval target for the batch pattern; not reproduced here, referenced as the thing `labs/rag-evaluation/ragas-scripts/Evaluate_RAG_PyTest.py` calls.

## RAGAS's unit: `SingleTurnSample`, not `LLMTestCase`

Where DeepEval's unit of evaluation is an `LLMTestCase` (`input`, `actual_output`, `retrieval_context`, ...), RAGAS's is a `SingleTurnSample` — same idea, different field names (`user_input`, `response`, `retrieved_contexts`, `reference`). And where DeepEval takes a `model=` string and resolves the judge model internally, RAGAS wants an explicit judge object built by `llm_factory(model, client=...)` — a small extra step, but `provider=` lets the same call use Anthropic, Gemini, Groq and others as the judge.

This notebook uses the RAGAS 0.4 API: metrics come from `ragas.metrics.collections` and are scored with `await metric.ascore(...)`. Older tutorials import from `ragas.metrics` and call `single_turn_ascore`; that path is deprecated.

In [ ]:
# ============ SETUP ============
import os

from dotenv import load_dotenv, find_dotenv
from openai import AsyncOpenAI
from ragas import SingleTurnSample
from ragas.llms import llm_factory

load_dotenv(find_dotenv())
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY (shell env or .env) before running this notebook."

# RAGAS 0.4: llm_factory builds the judge from a model name plus an OpenAI client.
# (The 0.3 pattern, LangchainLLMWrapper(ChatOpenAI(...)), is deprecated.)
evaluator_llm = llm_factory("gpt-4o-mini", client=AsyncOpenAI())

print("RAGAS evaluator ready")

### Quickstart: a single metric on a single sample

In [ ]:
# ============ QUICKSTART: ONE METRIC, ONE SAMPLE ============
from ragas.metrics.collections import ContextPrecisionWithoutReference

sample = SingleTurnSample(
    user_input="Where is the Eiffel Tower located?",
    response="The Eiffel Tower is located in Paris.",
    retrieved_contexts=["The Eiffel Tower is located in Paris, France."],
)

context_precision = ContextPrecisionWithoutReference(llm=evaluator_llm)
# Collections metrics take the fields as keyword arguments and return a MetricResult.
result = await context_precision.ascore(
    user_input=sample.user_input,
    response=sample.response,
    retrieved_contexts=sample.retrieved_contexts,
)
print(f"Context Precision (no reference): {result.value:.2f}")

Note the `WithoutReference` suffix — RAGAS is explicit in its class names about which of its metrics need a reference/ground-truth answer and which don't, the same reference-based/referenceless split from Module 0 §2, just spelled out in the class name instead of inferred from which arguments you pass.

### Multiple metrics, one sample — mirroring Module 1.2/1.3's retriever + generator split

In [ ]:
# ============ MULTIPLE RAGAS METRICS TOGETHER ============
from ragas.metrics.collections import ContextPrecisionWithoutReference, ContextRecall, Faithfulness

# A sample where retrieval pulled in one irrelevant chunk alongside the correct one --
# structured to make Precision and Recall diverge, the same way Module 1.2's paired
# examples did for DeepEval's ContextualPrecision/ContextualRecall.
sample = SingleTurnSample(
    user_input="Where is the Eiffel Tower located?",
    response="The Eiffel Tower is located in Paris.",
    reference="The Eiffel Tower is located in Paris. France is a country in Europe.",
    retrieved_contexts=[
        "India is in Asia.",  # irrelevant -- hurts precision
        "The Eiffel Tower is located in Paris.",  # relevant
    ],
)

context_recall = ContextRecall(llm=evaluator_llm)
context_precision = ContextPrecisionWithoutReference(llm=evaluator_llm)
faithfulness = Faithfulness(llm=evaluator_llm)

# Each metric names exactly the fields it reads -- recall needs the reference,
# the other two don't.
recall = await context_recall.ascore(
    user_input=sample.user_input,
    retrieved_contexts=sample.retrieved_contexts,
    reference=sample.reference,
)
precision = await context_precision.ascore(
    user_input=sample.user_input,
    response=sample.response,
    retrieved_contexts=sample.retrieved_contexts,
)
faithful = await faithfulness.ascore(
    user_input=sample.user_input,
    response=sample.response,
    retrieved_contexts=sample.retrieved_contexts,
)

print(f"Context Recall Score:    {recall.value:.2f}")
print(f"Context Precision Score: {precision.value:.2f}")
print(f"Faithfulness Score:      {faithful.value:.2f}")

**Reading the output:** this is the same retriever/generator split from the rest of Module 1, in RAGAS's vocabulary — `ContextRecall`/`ContextPrecisionWithoutReference` are RAGAS's answers to Module 1.2's Contextual Recall/Precision, and `Faithfulness` here is RAGAS's version of Module 1.3's `FaithfulnessMetric`. Expect Context Precision to come back lower than Context Recall on this sample specifically because of the irrelevant "India is in Asia" chunk sitting alongside the correct one — precision penalizes that noise directly, while recall (checked against `reference`, which the irrelevant chunk doesn't contradict) is less affected by it. All three metrics run as independent `async` calls here (`await metric.ascore(...)`, which works at the top level of a Jupyter cell) — RAGAS is async-first, unlike DeepEval's default synchronous `.measure()`.

### The batch/CI pattern: score a live system, write results back to a spreadsheet

The examples above score one hand-built sample at a time — fine for learning the API, not how RAGAS gets used against a real system. `labs/rag-evaluation/ragas-scripts/Evaluate_RAG_PyTest.py` shows the production shape: a `pytest`-parametrized test that reads questions from an Excel sheet, calls a **live RAG backend** for each one, scores the response with RAGAS, and writes the score back into the same spreadsheet. This is the same "batch-score against a live app, results land in a spreadsheet" pattern you'll see again in Module 5's CrewAI capstone, just with RAGAS instead of DeepEval's `TaskCompletionMetric` as the scorer. Reproduced here, updated to the RAGAS 0.4 API (the lab script still uses the 0.3 one), as a pattern to recognize, not to run as-is (it points at a hardcoded local path and `localhost:8000`, specific to the machine it was written on):

```python
import pytest
import pandas as pd
import asyncio
import requests
from ragas import SingleTurnSample
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecisionWithoutReference

EXCEL_FILE = "test_questions.xlsx"
BACKEND_URL = "http://localhost:8000"
evaluator_llm = llm_factory("gpt-4o-mini", client=AsyncOpenAI())

@pytest.mark.parametrize("row", pd.read_excel(EXCEL_FILE).iterrows())
def test_context_precision(row):
    index, data = row
    question = data["question"]

    # 1. Ask the live RAG backend
    response = requests.post(f"{BACKEND_URL}/ask", data={"prompt": question}).json()

    # 2. Score with RAGAS
    sample = SingleTurnSample(
        user_input=question,
        response=response["answer"],
        retrieved_contexts=[response["retrieved_docs"][0]],
    )
    score = asyncio.run(
        ContextPrecisionWithoutReference(llm=evaluator_llm).ascore(
            user_input=sample.user_input,
            response=sample.response,
            retrieved_contexts=sample.retrieved_contexts,
        )
    ).value

    # 3. Write the score back to the spreadsheet, next to the question it belongs to
    df = pd.read_excel(EXCEL_FILE)
    df.at[index, "score"] = score
    df.to_excel(EXCEL_FILE, index=False)

    # 4. Assert as a CI gate
    assert 0.0 <= score <= 1.0
```

The shape worth internalizing: `pytest.mark.parametrize` turns "one eval sample" into "one test per row of a spreadsheet," so `pytest` itself becomes the batch runner and the pass/fail report — no separate eval-harness loop to write. This is a general pattern, not RAGAS-specific; you'll see it again with DeepEval's `TaskCompletionMetric` in Module 5.

## Summary

- RAGAS's unit is `SingleTurnSample` (`user_input`, `response`, `retrieved_contexts`, `reference`) — conceptually the same as DeepEval's `LLMTestCase`, different field names, and an explicit `llm_factory(...)` step to supply the judge model.
- RAGAS is async-first (`await metric.ascore(...)` on metrics from `ragas.metrics.collections`) rather than DeepEval's default synchronous `.measure()`.
- The metric names spell out the reference-based/referenceless distinction directly in the class name (`...WithoutReference`) rather than inferring it from which fields you populate.
- The `pytest.mark.parametrize` + live-backend + spreadsheet-scoring pattern shown here is the same batch-evaluation shape you'll see again in Module 5, just with a different metric library doing the scoring.
- Next: [Module 1.7](07_RAG_Capstone_Build_and_Evaluate.ipynb) — the capstone. A real RAG system, a synthetic golden dataset, and the full metric suite (DeepEval, spanning everything Modules 1.1–1.5 covered) run end to end.